# PhantomLM — Prototype Notebook
**16.7M · TinyStories → Alpaca · ~3 hours total**

Set Accelerator → GPU T4 x1 before running.

In [ ]:
# Cell 1: Clone repo + load modules
import os, sys, importlib.util
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_ALLOC_CONF']   = 'expandable_segments:True'
os.system('rm -rf /kaggle/working/MobileLLM')
os.system('git clone https://github.com/opszx/MobileLLM.git /kaggle/working/MobileLLM')
REPO = '/kaggle/working/MobileLLM'
def load_mod(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    m    = importlib.util.module_from_spec(spec)
    sys.modules[name] = m
    spec.loader.exec_module(m)
    return m
load_mod('bitlinear',   f'{REPO}/bitlinear.py')
load_mod('mamba_block', f'{REPO}/mamba_block.py')
load_mod('attention',   f'{REPO}/attention.py')
load_mod('moe',         f'{REPO}/moe.py')
cfg_m   = load_mod('config', f'{REPO}/config.py')
mdl_m   = load_mod('model',  f'{REPO}/model.py')
PhantomLMConfig = cfg_m.PhantomLMConfig
PhantomLM       = mdl_m.PhantomLM
print('Repo cloned and modules loaded OK')

In [ ]:
# Cell 2: Install deps
os.system('pip install einops datasets transformers sentencepiece -q')
print('Done')

In [ ]:
# Cell 3: GPU check
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory/1024**3:.1f}GB')
else:
    print('No GPU — enable T4 in Settings')

In [ ]:
# Cell 4: Tokenizer FIRST
from transformers import AutoTokenizer
tokenizer           = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE          = tokenizer.vocab_size
print(f'Tokenizer: GPT-2  vocab={VOCAB_SIZE}')

In [ ]:
# Cell 5: Build model
# Tiny = 16.7M params for prototype. Switch to phantom_350m() for full run.
config = PhantomLMConfig.phantom_tiny()
config.vocab_size        = VOCAB_SIZE  # MUST match tokenizer
config.max_seq_len       = 128         # 128 enables coherent text (was 64)
config.batch_size        = 16
config.d_state           = 8
config.moe_aux_loss_coef = 0.001       # fixed: was 0.01, caused expert collapse
config.max_steps         = 2000
config.warmup_steps      = 200
config.learning_rate     = 3e-4
config.grad_clip         = 1.0
model  = PhantomLM(config).to(device).to(torch.bfloat16)
params = sum(p.numel() for p in model.parameters())
used   = torch.cuda.memory_allocated()/1024**3
print(f'Parameters: {params:,}  VRAM: {used:.2f}GB')
x = torch.randint(0, config.vocab_size, (2, 32), device=device)
loss, _, lm, aux = model(x, targets=x, use_checkpoint=True)
print(f'Sanity check OK  loss={lm.item():.4f}  aux={aux.item():.4f}')
del x; torch.cuda.empty_cache()

In [ ]:
# Cell 6: TinyStories dataset (Stage 1)
from datasets import load_dataset
from torch.utils.data import IterableDataset
print('Loading TinyStories...')
stories = load_dataset('roneneldan/TinyStories', split='train', streaming=True)
class TinyStoriesDataset(IterableDataset):
    def __init__(self, ds, tok, seq_len, max_tok=20_000_000):
        self.ds, self.tok, self.seq_len, self.max_tok = ds, tok, seq_len, max_tok
        self.eos = tok.eos_token_id
    def __iter__(self):
        buf, total = [], 0
        for item in self.ds:
            if total >= self.max_tok: return
            text = item.get('text','').strip()
            if not text: continue
            toks = self.tok.encode(text) + [self.eos]
            buf += toks; total += len(toks)
            while len(buf) >= self.seq_len+1:
                chunk = buf[:self.seq_len+1]; buf = buf[self.seq_len:]
                yield (torch.tensor(chunk[:-1], dtype=torch.long),
                       torch.tensor(chunk[1:],  dtype=torch.long))
train_dataset = TinyStoriesDataset(stories, tokenizer, config.max_seq_len, 20_000_000)
print('TinyStories ready — 20M tokens')

In [ ]:
# Cell 7: Trainer
import math, time
from torch.optim import AdamW
from torch.utils.data import DataLoader
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
class PhantomTrainer:
    def __init__(self, model, config, train_dataset, checkpoint_dir=CHECKPOINT_DIR, grad_accum=4, log_every=50, save_every=500):
        self.model=model; self.config=config; self.ckpt_dir=checkpoint_dir
        self.grad_acc=grad_accum; self.log_every=log_every; self.save_every=save_every
        self.step=0; self.best_loss=float('inf')
        self.loader=DataLoader(train_dataset, batch_size=config.batch_size, num_workers=2, pin_memory=True)
        self.optimizer=AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.1, betas=(0.9,0.95))
    def get_lr(self, step):
        max_lr=self.config.learning_rate; min_lr=max_lr*0.1
        w,T=self.config.warmup_steps,self.config.max_steps
        if step<w: return max_lr*step/max(w,1)
        if step>=T: return min_lr
        return min_lr+0.5*(max_lr-min_lr)*(1+math.cos(math.pi*(step-w)/(T-w)))
    def save(self, tag):
        path=os.path.join(self.ckpt_dir,f'phantomlm_{tag}.pt')
        torch.save({'step':self.step,'model_state_dict':self.model.state_dict(),'optimizer_state_dict':self.optimizer.state_dict(),'config':self.config,'best_loss':self.best_loss}, path)
        print(f'  Saved: {path}')
        if tag.startswith('step_'):
            old=os.path.join(self.ckpt_dir,f'phantomlm_step_{self.step-self.save_every}.pt')
            if os.path.exists(old): os.remove(old); print(f'  Deleted: {old}')
    def resume(self, path):
        if not os.path.exists(path): print('No checkpoint — fresh start'); return
        ckpt=torch.load(path,map_location='cpu',weights_only=False)
        self.model.load_state_dict(ckpt['model_state_dict'])
        self.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        self.step=ckpt['step']; self.best_loss=ckpt.get('best_loss',float('inf'))
        print(f'Resumed from step {self.step}')
    def train(self):
        self.model.train(); self.optimizer.zero_grad()
        t0,total_tokens=time.time(),0
        for batch_idx,(x,y) in enumerate(self.loader):
            if self.step>=self.config.max_steps:
                print('Training complete.'); self.save('final'); return
            x,y=x.to(device),y.to(device)
            for pg in self.optimizer.param_groups: pg['lr']=self.get_lr(self.step)
            with torch.autocast(device_type='cuda',dtype=torch.bfloat16):
                loss,_,lm_loss,aux_loss=self.model(x,targets=y,use_checkpoint=True)
            (loss/self.grad_acc).backward(); total_tokens+=x.numel()
            if (batch_idx+1)%self.grad_acc==0:
                gn=torch.nn.utils.clip_grad_norm_(self.model.parameters(),self.config.grad_clip)
                self.optimizer.step(); self.optimizer.zero_grad(); torch.cuda.empty_cache(); self.step+=1
                if self.step%self.log_every==0:
                    tok_s=total_tokens/(time.time()-t0); vram=torch.cuda.memory_allocated()/1024**3
                    print(f'Step {self.step:5d} | loss:{lm_loss.item():.4f} | aux:{aux_loss.item():.4f} | lr:{self.get_lr(self.step):.2e} | norm:{gn:.3f} | tok/s:{tok_s:,.0f} | VRAM:{vram:.1f}GB')
                    t0,total_tokens=time.time(),0
                if self.step%self.save_every==0:
                    self.save(f'step_{self.step}')
                    if lm_loss.item()<self.best_loss: self.best_loss=lm_loss.item(); self.save('best')
print('Trainer ready')

In [ ]:
# Cell 8: Stage 1 — TinyStories pretraining
# Expected time: ~1.5 hours  Target loss: ~3.5-4.5
config.max_steps=2000; config.warmup_steps=200
trainer=PhantomTrainer(model,config,train_dataset,CHECKPOINT_DIR,grad_accum=4,log_every=50,save_every=500)
trainer.resume(f'{CHECKPOINT_DIR}/phantomlm_best.pt')
print(f'Stage 1: step {trainer.step} → {config.max_steps}')
trainer.train()

In [ ]:
# Cell 9: Generation test after Stage 1
# Should produce coherent story continuations
model.eval()
def generate(prompt, max_new=100, temp=0.8):
    toks=tokenizer.encode(prompt)
    ids=torch.tensor([toks],device=device)
    with torch.no_grad():
        out=model.generate(ids,max_new_tokens=max_new,temperature=temp,eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0,len(toks):].tolist(),skip_special_tokens=True)
for p in ['Once upon a time there was a','The little dog ran to','One day a girl named Lily']:
    print(f'Prompt  : {p}'); print(f'Response: {generate(p)}'); print('-'*60)
model.train()

In [ ]:
# Cell 10: Alpaca dataset (Stage 2)
from datasets import load_dataset as lds
from torch.utils.data import Dataset as TorchDataset
import copy
alpaca=lds('yahma/alpaca-cleaned',split='train').select(range(3000))
class AlpacaDataset(TorchDataset):
    def __init__(self, data, tok, seq_len):
        self.seq_len=seq_len; self.eos=tok.eos_token_id; self.data=[]
        for item in data:
            ctx=f'### Input:\n{item["input"]}\n' if item.get('input','').strip() else ''
            text=f'### Instruction:\n{item["instruction"]}\n{ctx}### Response:\n{item["output"]}'
            toks=tok.encode(text,truncation=True,max_length=seq_len)+[self.eos]
            if len(toks)>8:
                while len(toks)<seq_len+1: toks.append(self.eos)
                self.data.append(toks[:seq_len+1])
    def __len__(self): return len(self.data)
    def __getitem__(self,i):
        t=self.data[i]; return (torch.tensor(t[:self.seq_len],dtype=torch.long),torch.tensor(t[1:self.seq_len+1],dtype=torch.long))
ft_dataset=AlpacaDataset(alpaca,tokenizer,config.max_seq_len)
print(f'Alpaca examples: {len(ft_dataset):,}')

In [ ]:
# Cell 11: Stage 2 — Alpaca fine-tuning
# Expected time: ~30 mins  Target loss: ~2.5-3.5
ft_config=copy.deepcopy(config)  # deep copy — NOT a reference
ft_config.learning_rate=5e-5; ft_config.max_steps=trainer.step+600; ft_config.warmup_steps=50
ft_trainer=PhantomTrainer(model,ft_config,ft_dataset,CHECKPOINT_DIR,grad_accum=2,log_every=20,save_every=200)
ft_trainer.step=trainer.step
print(f'Stage 2: step {ft_trainer.step} → {ft_config.max_steps}')
ft_trainer.train()

In [ ]:
# Cell 12: Final generation test
model.eval()
for label,prompt in [
    ('Story', 'Once upon a time there was a'),
    ('Fact',  'The capital of France is'),
    ('Math',  '1 + 1 ='),
    ('QA',    '### Instruction:\nWhat is artificial intelligence?\n### Response:\n'),
    ('QA',    '### Instruction:\nExplain photosynthesis simply.\n### Response:\n'),
    ('Code',  '### Instruction:\nWrite a Python function to sort a list.\n### Response:\n'),
]:
    r=generate(prompt,max_new=120,temp=0.7)
    print(f'[{label}] {prompt.split(chr(10))[0]}')
    print(f'→ {r[:200]}')
    print('-'*60)

In [ ]:
# Cell 13: Efficiency benchmark — Table 2 of paper
import time; model.eval()
print(f'{"Context":>10} {"Tok/s":>10} {"Latency":>12} {"VRAM MB":>10}')
print('-'*46)
for seq_len in [16,32,64,128]:
    if seq_len>config.max_seq_len: continue
    dummy=torch.randint(0,config.vocab_size,(1,seq_len),device=device)
    with torch.no_grad(): _=model(dummy,return_loss=False)
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0=time.perf_counter()
    with torch.no_grad():
        for _ in range(50): _=model(dummy,return_loss=False)
    torch.cuda.synchronize(); elapsed=time.perf_counter()-t0
    print(f'{seq_len:>10} {seq_len*50/elapsed:>10.0f} {elapsed/50*1000:>11.1f}ms {torch.cuda.max_memory_allocated()/1024**2:>10.1f}')
params=sum(p.numel() for p in model.parameters())
print(f'\nModel size (bfloat16): {params*2/1024**2:.1f} MB')

In [ ]:
# Cell 14: Save results for paper
import json
results={'model':'PhantomLM-tiny','params':sum(p.numel() for p in model.parameters()),'total_steps':ft_trainer.step}
with open('/kaggle/working/paper_results.json','w') as f: json.dump(results,f,indent=2)
print('Saved to paper_results.json')
print(json.dumps(results,indent=2))